# Итоговое практическое задание
## Расширенный анализ телеметрии БАС
Сырой датасет: `data_final/telemetry_final_raw.csv`


**Задача:** выполнить полный цикл обработки, анализа и визуализации данных полёта.

## 0. Импорт библиотек и настройка путей

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import geopandas as gpd
import contextily as ctx

## 1. Загрузка и предобработка данных
Шаги:
- загрузка сырых данных;
- приведение времени к `datetime`;
- сортировка и устранение дублей;
- базовая фильтрация аномалий по диапазонам;
- восстановление пропусков и выравнивание временной сетки.

In [ ]:
df = pd.read_csv("telemetry_final_raw.csv")

print("raw dataset size and dimensions:", df.shape)
print("\ncols:")
print(df.columns.tolist())
print("\ndata types:")
print(df.dtypes)
print("\nmissing data:")
print(df.isna().sum())

In [ ]:
anomalies = (
    (~df["lat"].between(-90, 90)) |
    (~df["lon"].between(-180, 180)) |
    (df["alt_m"] < 0) |
    (df["speed_mps"] < 0) |
    (df["voltage_v"] <= 0)
).sum()

In [ ]:
df["time"] = pd.to_datetime(df["time"], errors="coerce")

df = df.dropna(subset=["time"])

df = df.sort_values("time")

df = df.drop_duplicates()

df = df.drop_duplicates(subset=["time"], keep="first")

df = df.reset_index(drop=True)

print("dataset size and dimensions after cleansing:", df.shape)
print("period:", df["time"].min(), "-", df["time"].max())

In [ ]:
time_diff = df["time"].diff().dt.total_seconds()

print("\nintervals statistics:")
print(time_diff.describe())

print("\nthe most frequent intervals:")
print(time_diff.value_counts().head(10))

In [ ]:
df.loc[df["lat"].notna() & ~df["lat"].between(-90, 90), "lat"] = np.nan
df.loc[df["lon"].notna() & ~df["lon"].between(-180, 180), "lon"] = np.nan

df.loc[df["alt_m"] < 0, "alt_m"] = np.nan
df.loc[df["speed_mps"] < 0, "speed_mps"] = np.nan
df.loc[df["voltage_v"] <= 0, "voltage_v"] = np.nan

print("missing entries after conforming to physical limitations:")
print(df.isna().sum())

In [ ]:
df = df.set_index("time").resample("1s").asfreq()

numeric_cols = ["lat", "lon", "alt_m", "speed_mps", "voltage_v"]

df[numeric_cols] = df[numeric_cols].interpolate(method="time", limit_direction="both")

df = df.reset_index()

print("post resampling shape:", df.shape)
print("\npost interpolation missing entries:")
print(df.isna().sum())

### Комментарий по предобработке

Данные были преобразованы к формату datetime, отсортированы и очищены от дубликатов. Некорректные значения координат, высоты и скорости были заменены на пропуски согласно физическим ограничениям.

В исходном временном ряду 3439 интервалов составляют 1 секунду и 80 — 2 секунды, поэтому данные были приведены к регулярной сетке с шагом 1 секунда. Пропущенные значения восстановлены линейной интерполяцией по времени.

## 2. Расчёт ключевых показателей
Необходимо вычислить:
- Среднюю и максимальную скорость
- Минимальную и максимальную высоту
- Продолжительность полёта
- количество аномалий (на сырых данных);
- вертикальную скорость и её характеристики.


In [ ]:
mean_speed = df["speed_mps"].mean()
max_speed = df["speed_mps"].max()

min_alt = df["alt_m"].min()
max_alt = df["alt_m"].max()

flight_duration = df["time"].iloc[-1] - df["time"].iloc[0]

df["vertical_speed_mps"] = df["alt_m"].diff() / df["time"].diff().dt.total_seconds()

mean_vertical_speed = df["vertical_speed_mps"].mean()
max_vertical_speed = df["vertical_speed_mps"].max()
min_vertical_speed = df["vertical_speed_mps"].min()

print(f"avg spd: {mean_speed:.2f} m/s")
print(f"max spd: {max_speed:.2f} m/s")
print(f"min alt: {min_alt:.2f} m")
print(f"max alt: {max_alt:.2f} m")
print(f"flight duration: {flight_duration}")
print(f"avg vert spd: {mean_vertical_speed:.2f} m/s")
print(f"max vert spd: {max_vertical_speed:.2f} m/s")
print(f"min vert spd: {min_vertical_speed:.2f} m/s")
print(f"anomalies number: {anomalies}")

## 3. Простое выделение фаз полёта
Простейшее правил-based деление по высоте и вертикальной скорости:
- `ground`: высота < 5 м;
- `takeoff_landing`: высота 5–20 м и |climb_rate| > 0.3 м/с;
- `climb`: высота ≥ 20 м, climb_rate > 0.3 м/с;
- `descent`: высота ≥ 20 м, climb_rate < -0.3 м/с;
- `cruise`: высота ≥ 20 м, |climb_rate| ≤ 0.3 м/с.

In [ ]:
def classify_phase(row):
    alt = row["alt_m"]
    climb_rate = row["vertical_speed_mps"]

    if alt < 5:
        return "ground"
    elif 5 <= alt < 20 and abs(climb_rate) > 0.3:
        return "takeoff_landing"
    elif alt >= 20 and climb_rate > 0.3:
        return "climb"
    elif alt >= 20 and climb_rate < -0.3:
        return "descent"
    elif alt >= 20 and abs(climb_rate) <= 0.3:
        return "cruise"
    else:
        return "takeoff_landing"
    
df["phase"] = df.apply(classify_phase, axis=1)

print(df["phase"].value_counts())

## 4. Визуализации
Строим:
- траекторию по GPS;
- скорость во времени;
- высоту во времени;
- вертикальную скорость во времени;
- зависимость напряжения от скорости.

In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

gdf = gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(10, 8))

for phase, group in gdf.groupby("phase"):
    ax.plot(
        group.geometry.x,
        group.geometry.y,
        label=phase,
        zorder=2
    )

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, alpha=0.3, zorder=0)

ax.set_title("gps-based flight path")
ax.set_xlabel("longtitude")
ax.set_ylabel("latitude")
ax.legend()
ax.set_axis_off()

plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for phase, group in df.groupby("phase"):
    plt.plot(group["lon"], group["lat"], label=phase)

plt.xlabel("longtitude")
plt.ylabel("latitude")
plt.title("gps-based flight path")
plt.legend()
plt.grid()
plt.show()

In [ ]:
phase_colors = {
    "ground": "gray",
    "takeoff_landing": "orange",
    "climb": "green",
    "descent": "red",
    "cruise": "blue"
}

params = {
    "speed_mps": ("speed", "m/s"),
    "alt_m": ("altitude", "m"),
    "vertical_speed_mps": ("vertical speed", "m/s")
}

fig = go.Figure()

for param, (name, unit) in params.items():
    for phase, group in df.groupby("phase"):
        fig.add_trace(go.Scatter(
            x=group["time"],
            y=group[param],
            mode="lines",
            name=phase,
            legendgroup=phase,
            line=dict(color=phase_colors[phase]),
            visible=(param == "speed_mps"),
            hovertemplate=f"%{{x}}<br>{name}: %{{y:.2f}} {unit}<extra>{phase}</extra>"
        ))

buttons = []

for i, (param, (name, unit)) in enumerate(params.items()):
    visible = []
    for p in params:
        visible += [p == param] * len(df["phase"].unique())

    buttons.append(dict(
        label=name,
        method="update",
        args=[
            { "visible": visible },
            { "yaxis": { "title": unit },
             "title": f"{name} by time" }
        ]
    ))

fig.update_layout(
    title="speed by time",
    xaxis_title="time",
    yaxis_title="m/s",
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1.0,
        y=1.15
    )],
    hovermode="x unified"
)

fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="speed_mps",
    y="voltage_v",
    color="phase",
    opacity=0.6,
    labels={
        "speed_mps": "speed, m/s",
        "voltage_v": "voltage, v",
        "phase": "phase"
    },
    title="voltage by speed relation"
)

fig.update_layout(legend_title="flight phase")

fig.show()

## 5. Вывод результатов анализа
- очищенный временной ряд
- метрики 
- статистика фаз 


In [ ]:
df

In [ ]:
df.describe()

In [ ]:
phase_stats = df.groupby("phase")[
    ["speed_mps", "alt_m", "vertical_speed_mps", "voltage_v"]
].describe()

phase_stats["alt_m"]

---
## Итоговые комментарии (пример)
1. На уровне предобработки телеметрия приведена к равномерной временной сетке, а выбросы по координатам и скорости сглажены.
2. Рассчитанные метрики (скорости, высоты, вертикальные скорости) позволяют количественно описать профиль полёта.
3. Простая модель фаз полёта даёт представление о структуре миссии (доли набора, крейсерского участка и снижения).
4. Результаты сохранены в машиночитаемом формате и могут быть использованы далее (кластеризация полётов, обнаружение аномалий и т.п.).

1. На этапе предобработки телеметрия была очищена от дубликатов и некорректных значений, приведена к равномерной временной сетке с шагом 1 секунда. Пропущенные значения восстановлены интерполяцией по времени.

2. Рассчитанные показатели скорости, высоты и вертикальной скорости позволяют количественно описать основные характеристики полёта.

3. Простое выделение фаз по высоте и вертикальной скорости позволяет определить основные этапы полёта: нахождение на земле, набор высоты, крейсерский участок и снижение.

4. Полученные результаты представлены в виде таблиц и интерактивных визуализаций и могут использоваться для дальнейшего анализа телеметрии, например для обнаружения аномалий или классификации полётов.